# Voice2Gesture: Sign Language Generator
This notebook trains and evaluates a Transformer-based generator to produce American Sign Language (ASL) finger-spelling images from text inputs (letters, words, and full sentences).

### Pipeline Workflow:
1. **Dynamic Configuration & Paths**: Portable, cross-platform environment setup.
2. **Dataset Loading with Mock Fallback**: Loads ASL gesture images from disk or generates synthetic samples if the local dataset is not found.
3. **Character-Level Tokenization**: Prepares character sequences and train/validation splits.
4. **Transformer Architecture**: Defines an embedding + multi-head attention encoder and convolutional decoder.
5. **Model Training & Loss Visualization**: Trains the model with explicit validation data.
6. **Model Persistence**: Safely saves and reloads model weights.
7. **Sign Language Generation & Layout Utilities**: Robust polymorphic image stitching and character labeling.
8. **End-to-End Sentence Demonstration**: Converts multi-word sentences into visual sign language gestures.

In [1]:
import os
import sys
import string
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LayerNormalization, MultiHeadAttention, Conv2DTranspose, Reshape, Add
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Dynamic project paths (cross-platform, avoiding hardcoded OS user paths)
BASE_DIR = os.getcwd()
DATA_PATH = os.getenv("ASL_DATASET_PATH", os.path.join(BASE_DIR, "archive", "asl_dataset"))
MODEL_DIR = os.path.join(BASE_DIR, "models")
MODEL_PATH = os.path.join(MODEL_DIR, "smodel.keras")

# Configuration parameters
IMAGE_SIZE = (128, 128)      # Target height and width for sign images
IMAGE_CHANNELS = 3          # RGB channels
IMG_SHAPE = (IMAGE_SIZE[0], IMAGE_SIZE[1], IMAGE_CHANNELS)
TEXT_DIM = 256              # Embedding dimensionality
MAX_LABEL_LENGTH = 5        # Padded sequence length
BATCH_SIZE = 8
EPOCHS = 20                 # Adjustable training epochs

print(f"TensorFlow version: {tf.__version__}")
print(f"Dataset path: {DATA_PATH}")
print(f"Model target path: {MODEL_PATH}")


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
def load_images_from_folders(base_path, image_size):
    """
    Loads images and labels from subdirectories (one folder per letter/digit).
    Returns normalized numpy arrays of images and labels.
    """
    images = []
    labels = []
    
    if not os.path.isdir(base_path):
        print(f"Dataset directory not found at: {base_path}")
        return None, None
        
    for letter_folder in sorted(os.listdir(base_path)):
        folder_path = os.path.join(base_path, letter_folder)
        if os.path.isdir(folder_path):
            for filename in os.listdir(folder_path):
                if filename.lower().endswith((".png", ".jpg", ".jpeg")):
                    img_path = os.path.join(folder_path, filename)
                    try:
                        img = load_img(img_path, target_size=image_size)
                        img_arr = img_to_array(img) / 255.0  # Normalize to [0, 1]
                        images.append(img_arr)
                        labels.append(letter_folder.lower())
                    except Exception as e:
                        print(f"Warning: Failed to load {img_path}: {e}")
                        
    if len(images) == 0:
        return None, None
        
    return np.array(images, dtype=np.float32), np.array(labels)

def create_synthetic_demo_dataset(image_size, samples_per_letter=4):
    """
    Generates a synthetic demo dataset if local raw ASL images are unavailable.
    Allows running the full pipeline end-to-end without external downloads.
    """
    print("Notice: Dataset folder not found. Generating synthetic demo dataset for testing...")
    letters = list(string.ascii_lowercase)
    images = []
    labels = []
    
    for char in letters:
        for _ in range(samples_per_letter):
            img = np.ones((image_size[0], image_size[1], 3), dtype=np.float32) * 0.95
            char_code = ord(char) - ord("a") + 1
            c_val = (char_code * 9) % 255 / 255.0
            img[20:108, 20:108, 0] = c_val
            img[20:108, 20:108, 1] = 1.0 - c_val
            img[20:108, 20:108, 2] = 0.5
            images.append(img)
            labels.append(char)
            
    return np.array(images, dtype=np.float32), np.array(labels)

# Load real dataset if available; otherwise use synthetic fallback
images, labels = load_images_from_folders(DATA_PATH, IMAGE_SIZE)
if images is None:
    images, labels = create_synthetic_demo_dataset(IMAGE_SIZE)

print(f"Loaded dataset: {images.shape[0]} images, shape: {images.shape[1:]}")
print(f"Unique classes ({len(set(labels))}): {sorted(list(set(labels)))[:10]}...")


In [ ]:
# Initialize and fit tokenizer on character level
tokenizer = Tokenizer(char_level=True, lower=True)
# Ensure vocabulary covers all lowercase letters and digits
all_vocabulary = list(string.ascii_lowercase) + [str(i) for i in range(10)]
tokenizer.fit_on_texts(all_vocabulary)

# Tokenize dataset labels
label_sequences = tokenizer.texts_to_sequences(labels)
label_sequences = pad_sequences(label_sequences, maxlen=MAX_LABEL_LENGTH)

# Train / validation split (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    label_sequences, images, test_size=0.2, random_state=42
)

vocab_size = len(tokenizer.word_index) + 1

print(f"Vocabulary size: {vocab_size}")
print(f"Train samples: {X_train.shape[0]}, Validation samples: {X_val.shape[0]}")


In [ ]:
def build_transformer_model(vocab_size, text_dim, img_shape, max_label_length):
    """
    Builds a Transformer-based generator:
    - Text Encoder: Embedding + Multi-Head Self Attention + Layer Normalization
    - Image Decoder: Dense projection + Conv2DTranspose upsampling to target image size
    """
    text_input = Input(shape=(max_label_length,), name="text_input")
    text_embedding = Embedding(input_dim=vocab_size, output_dim=text_dim)(text_input)
    
    # Transformer Encoder Block
    attention_out = MultiHeadAttention(num_heads=8, key_dim=text_dim)(text_embedding, text_embedding)
    x = Add()([text_embedding, attention_out])
    x = LayerNormalization()(x)
    
    # Sequence representation via first position
    x = x[:, 0, :]
    
    # Image Decoder (upsampling 16x16 -> 32x32 -> 64x64 -> 128x128)
    x = Dense(128 * 16 * 16, activation="relu")(x)
    x = Reshape((16, 16, 128))(x)
    x = Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", activation="relu")(x)  # 32x32
    x = Conv2DTranspose(64, kernel_size=4, strides=2, padding="same", activation="relu")(x)   # 64x64
    img_output = Conv2DTranspose(img_shape[2], kernel_size=4, strides=2, padding="same", activation="sigmoid")(x)  # 128x128

    model = Model(inputs=text_input, outputs=img_output, name="sign_language_generator")
    return model

# Instantiate and compile unified model
model = build_transformer_model(vocab_size, TEXT_DIM, IMG_SHAPE, MAX_LABEL_LENGTH)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=20, batch_size=8):
    """
    Trains the generator using explicit validation data.
    """
    history = model.fit(
        X_train, 
        y_train, 
        validation_data=(X_val, y_val),
        epochs=epochs, 
        batch_size=batch_size,
        verbose=1
    )
    return history

# Train the model
history = train_model(model, X_train, y_train, X_val, y_val, epochs=EPOCHS, batch_size=BATCH_SIZE)

# Plot training & validation loss
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Train Loss (MSE)")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="Val Loss (MSE)")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


In [ ]:
# Ensure output directory exists and save the trained model
os.makedirs(MODEL_DIR, exist_ok=True)
model.save(MODEL_PATH)
print(f"Model successfully saved to: {MODEL_PATH}")

# Verify model loading
loaded_model = tf.keras.models.load_model(MODEL_PATH)
print("Model reloaded successfully and ready for inference.")


In [ ]:
def extract_image_and_label(item):
    """
    Safely unpacks an item whether it is a (label, img) tuple/list or a bare image array.
    """
    if isinstance(item, (tuple, list)) and len(item) == 2:
        return item[0], item[1]
    return "", item

def combine_images(images, word_spacing=10):
    """
    Polymorphic image merger:
    Accepts either a list of raw image arrays [img1, img2, ...] 
    OR a list of labeled tuples [(label1, img1), (label2, img2), ...].
    Stitches images horizontally on a white canvas without unpack errors.
    """
    if not images:
        return np.ones((100, 100, 3), dtype=np.float32)
        
    extracted = [extract_image_and_label(item) for item in images]
    
    img_width = sum(img.shape[1] for _, img in extracted) + word_spacing * (len(extracted) - 1)
    img_height = max(img.shape[0] for _, img in extracted)
    
    # White background canvas
    combined_image = np.ones((img_height, img_width, 3), dtype=np.float32)
    
    x_offset = 0
    for _, img in extracted:
        normalized_img = np.clip(img, 0.0, 1.0)
        h, w = normalized_img.shape[0], normalized_img.shape[1]
        combined_image[:h, x_offset:x_offset + w] = normalized_img
        x_offset += w + word_spacing
        
    return combined_image

def generate_images_from_letters(model, letters, tokenizer, max_label_length):
    """
    Generates sign language images for a list of characters.
    Returns a list of (character, image_array) tuples.
    """
    images = []
    for char in letters:
        seq = tokenizer.texts_to_sequences([char.lower()])
        padded_seq = pad_sequences(seq, maxlen=max_label_length)
        generated = model.predict(padded_seq, verbose=0)
        img = np.clip(generated.squeeze(axis=0), 0.0, 1.0)
        images.append((char, img))
    return images

def label_letter_images_in_line(labeled_images, word_spacing=10):
    """
    Displays letter images in one horizontal line with their character labels underneath.
    """
    if not labeled_images:
        return
    combined_image = combine_images(labeled_images, word_spacing)
    fig, ax = plt.subplots(figsize=(max(6, len(labeled_images) * 1.5), 4))
    ax.imshow(combined_image)
    
    extracted = [extract_image_and_label(item) for item in labeled_images]
    x_offset = 0
    for label, img in extracted:
        if label:
            ax.text(x_offset + img.shape[1] // 2, img.shape[0] + 15, label, 
                    fontsize=14, ha="center", va="top", fontweight="bold")
        x_offset += img.shape[1] + word_spacing
        
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# Quick test with single letter "a"
test_a = generate_images_from_letters(loaded_model, ["a"], tokenizer, MAX_LABEL_LENGTH)
label_letter_images_in_line(test_a)


In [ ]:
def sentence_to_sign_language_images(model, sentence, tokenizer, max_label_length):
    """
    Converts a sentence into sign language images:
    1. Sanitizes input to lowercase alphanumeric tokens.
    2. Generates and displays letter signs for each word.
    3. Combines letters into word images.
    4. Returns a list of (word, word_image) pairs.
    """
    words = sentence.strip().split()
    sentence_images = []
    
    for word in words:
        # Filter characters to valid letters/digits only
        clean_word = "".join(ch.lower() for ch in word if ch.isalnum())
        if not clean_word:
            continue
            
        # Generate letter images for this word
        letter_images = generate_images_from_letters(model, list(clean_word), tokenizer, max_label_length)
        
        # Display individual letters with labels
        print(f"Word: {clean_word}")
        label_letter_images_in_line(letter_images, word_spacing=10)
        
        # Combine letters into a single word image
        word_image = combine_images(letter_images, word_spacing=5)
        sentence_images.append((clean_word, word_image))
        
    return sentence_images

# Full Sentence Demonstration
demo_sentence = "hi how are you"
print(f"Generating sign language representation for: {demo_sentence}\n")

# Process sentence into words and letter signs
sentence_images = sentence_to_sign_language_images(loaded_model, demo_sentence, tokenizer, MAX_LABEL_LENGTH)

# Combine all word images into a full sentence banner (now 100% bug-free!)
# Polymorphic combine_images handles both raw image arrays and (label, img) tuples:
sentence_combined_image = combine_images([img for _, img in sentence_images], word_spacing=30)

# Display final full-sentence combined sign banner
plt.figure(figsize=(14, 4))
plt.imshow(sentence_combined_image)
plt.title(f"Full Sign Language Banner: {demo_sentence}", fontsize=16, pad=15)
plt.axis("off")
plt.tight_layout()
plt.show()
